In [ ]:
import pandas as pd
import os

# --- Inställningar ---
# Filnamnet på din sparade prediktionsfil (i mappen 'predictions')
PREDICTION_FILENAME = 'predictions_all_20251028_1038.parquet'
PREDICTION_FILE_PATH = os.path.join('predictions', PREDICTION_FILENAME)

# Filnamnet på din faktiska datafil (i mappen 'data')
ACTUAL_DATA_PATH = os.path.join('data', 'trips_combined_total.parquet') 

# --- Läs in Data ---
print(f"Laddar in prediktionsfil: {PREDICTION_FILE_PATH}")
df_predictions = pd.read_parquet(PREDICTION_FILE_PATH)

print(f"Laddar in faktisk datafil: {ACTUAL_DATA_PATH}")
df_actual = pd.read_parquet(ACTUAL_DATA_PATH)

# --- Jämför Kolumner ---
print("\n--- Kolumner i Prediktionsfilen ---")
print(df_predictions.columns.tolist())

print("\n--- Kolumner i Faktisk Datafil ---")
print(df_actual.columns.tolist())

# Kontrollera om AdvertisedTrainIdent finns i båda
print("\n--- Kontroll av matchningskolumn ---")
match_col = 'AdvertisedTrainIdent'
in_pred = match_col in df_predictions.columns
in_actual = match_col in df_actual.columns

if in_pred and in_actual:
    print(f"✅ Kolumnen '{match_col}' finns i båda filerna.")
elif in_pred or in_actual:
    print(f"⚠️ Kolumnen '{match_col}' hittades inte i båda filerna!")
else:
    print(f"❌ Kolumnen '{match_col}' hittades inte alls.")

: 

In [ ]:
print(f"Antal rader i df_predictions: {df_predictions.shape[0]}")
print(f"Antal rader i df_actual: {df_actual.shape[0]}")

print(df_predictions.head())

print(df_actual.head())

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix, precision_score, recall_score
import numpy as np

# --- Inställningar ---
# Använder start_planned (finns i df_predictions) och AdvertisedTrainIdent
JOIN_KEY = ['AdvertisedTrainIdent', 'start_planned'] 
ACTUAL_DELAY_COL = 'DelayMinutes' 
DELAY_THRESHOLD = 6
PRED_COL = 'Predicted_Delay'

print("🚀 Startar validering baserat på TÅG-ID OCH STARTTID...")

# --- STEG 1: Förberedelse och Datatypkonvertering ---

# 1. Konvertera start_planned i df_predictions (om det behövs)
df_predictions['start_planned'] = pd.to_datetime(df_predictions['start_planned'])

# 2. RÄTTA TILL NAMN: Byt namn på matchningskolumnen i df_actual
# Vi använder 'DepartureAdvertised' från df_actual som matchning mot 'start_planned'
# i df_predictions.
df_actual = df_actual.rename(columns={'DepartureAdvertised': 'start_planned'})
df_actual['start_planned'] = pd.to_datetime(df_actual['start_planned'])
# NOTERA: 'TripStartDate' behövde inte konverteras här, jag tog bort den onödiga raden.

# 3. Skapa den faktiska målvariabeln i df_actual
if ACTUAL_DELAY_COL not in df_actual.columns:
    raise KeyError(f"❌ FEL: Kolumnen för faktisk försening ('{ACTUAL_DELAY_COL}') hittades inte i df_actual.")
    
df_actual['is_delayed_actual'] = (
    df_actual[ACTUAL_DELAY_COL] >= DELAY_THRESHOLD
).astype(int)

# --- STEG 2: Skapa Validerings-DataFrame genom Sammanslagning (Merge) ---

# Kolumner som ska behållas från df_actual. 'is_delayed' är troligen duplikat av 'is_delayed_actual', 
# men jag behåller den för att undvika fel om du använder den senare.
actual_cols_to_keep = ['Operator', 'TrainOwner', 'trip_typeoftraffic', 'departure_station', 
                       'arrival_station', 'end_station_county', 'DelayMinutes',
                       'is_delayed', 'is_delayed_actual'] + JOIN_KEY

# Slå ihop prediktioner och faktiska resultat baserat på den utökade JOIN_KEY
df_validation_matched = pd.merge(
    df_predictions, 
    df_actual[actual_cols_to_keep], 
    on=JOIN_KEY, 
    how='inner' 
)

total_matched_trips = len(df_validation_matched)
print(f"✅ Valideringsdataset skapat. Antal matchade unika resor: {total_matched_trips}")


# --- STEG 3 & 4: Utvärdering och Statistik (som tidigare) ---

# Fortsätt med den befintliga utvärderingskoden:

y_true = df_validation_matched['is_delayed_actual']
y_pred = df_validation_matched[PRED_COL]

# Confusion Matrix: [[TN, FP], [FN, TP]]
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

# Beräkna statistik
total_matched_trips = len(df_validation_matched)
total_actual_delay = tp + fn
total_predicted_delay = tp + fp
correctly_found = tp
missed_delay = fn
percent_found = (tp / total_actual_delay) * 100 if total_actual_delay > 0 else 0


print("\n" + "="*70)
print("🎯 VALIDERINGSRESULTAT: Matchande Tågresor")
print("="*70)
print(f"Totala antalet matchade tågresor: {total_matched_trips}")
print(f"Totala faktiska förseningar (>= {DELAY_THRESHOLD} min): {total_actual_delay}")
print("-" * 70)
print(f"Modellen predikterade förseningar totalt (TP + FP): {total_predicted_delay}")
print(f"Korrekt hittade förseningar (TP): {correctly_found}")
print(f"Procent av faktiska förseningar som hittades (Recall): {percent_found:.2f}%")
print("-" * 70)

# Ytterligare statistik
print(f"Precision (Klass 1): {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall (Klass 1):    {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1-score (Klass 1):  {f1_score(y_true, y_pred, zero_division=0):.4f}")
print("-" * 70)
print(f"Missade förseningar (FN): {missed_delay}")
print(f"Felaktiga larm (FP): {fp}")
print("="*70)

# Lägg till Is_Correct-kolumnen
df_validation_matched['Is_Correct'] = (
    df_validation_matched['is_delayed_actual'] == df_validation_matched[PRED_COL]
).astype(int)

print("\n--- Valideringsdata med Match-kolumn (Topp 10) ---")
display_cols_final = ['AdvertisedTrainIdent', 'start_planned', ACTUAL_DELAY_COL, 
                      'is_delayed_actual', PRED_COL, 'Delay_Probability', 'Is_Correct']
display(df_validation_matched[display_cols_final].head(10))